In [2]:
from cfb.config import ROOT, SQL_ANALYSIS_DIR
from cfb.db import engine, read_query
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# modeling (scikit-learn)
import joblib                                                   # save / load the trained model
from sklearn.linear_model import LogisticRegression             # baseline model
from sklearn.ensemble import HistGradientBoostingClassifier     # main model: handles NULLs, monotonic constraints
from sklearn.pipeline import (make_pipeline, Pipeline)
from sklearn.impute import SimpleImputer                        # fill NULLs for the baseline (logistic can't take them)
from sklearn.preprocessing import (StandardScaler,SplineTransformer)
from sklearn.isotonic import IsotonicRegression                 # calibrate probabilities on the validation season
from sklearn.calibration import calibration_curve, CalibrationDisplay
from sklearn.metrics import log_loss, brier_score_loss, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer

from sklearn.metrics import (
    classification_report,
    ConfusionMatrixDisplay,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_curve,
    roc_auc_score,
)

In [3]:
fg_query = r"""
WITH fg AS (
    SELECT
        f.*,
        coalesce(
            substring(p.play_text FROM '(?i)(\d+)\s*(?:yd|yard)s?\.?\s+(?:field goal|fg)'),
            substring(p.play_text FROM '(?i)(?:field goal|fg)\s+attempt\s+from\s+(\d+)')
        )::int AS raw_num,
        -- kicker's name, from whichever of the three play-text formats matches
        coalesce(
            -- 1. Jersey format:  "#47 T.Woody field goal attempt from 25 yards"
            substring(p.play_text FROM '(?i)#\d+\s+(\S+(?:\s+(?!field|fg)\S+)?)\s+(?:field goal|fg)'),
            -- 2. "by" format:    "22 yard field goal by Alex Trlica (TTU) is good."
            substring(p.play_text FROM '(?i)(?:field goal|fg)\s+(?:is\s+)?(?:good\s+)?by\s+([A-Za-z.''\-]+(?:\s+[A-Za-z.''\-]+){0,2}?)\s*(?:\(|,|$)'),
            -- 3. Name first:     "Chase Contreraz 22 Yd Field Goal", "Kenny Byrd field goal attempt from 40"
            substring(p.play_text FROM '(?i)^((?:[A-Za-z.''\-]+\s+){0,2}[A-Za-z.''\-]+)\s+(?:\d+\s*(?:yd|yard)s?\.?\s+)?(?:field goal|fg)')
        ) AS raw_name
    FROM marts.fourth_down_features f
    LEFT JOIN core.plays p
        ON p.play_id = f.play_id
    WHERE f.decision = 'field goal'
),
named AS (
    -- one key per kicker: first initial + last word, upper case ("Daniel Carlson" -> "D.CARLSON"),
    -- suffixes dropped. game_date for ordering (postseason weeks restart at 1).
    SELECT
        fg.*,
        upper(left(regexp_replace(raw_name, '[\s,]+(Jr\.?|Sr\.?|II|III|IV)$', '', 'i'), 1)) || '.' ||
        upper(substring(regexp_replace(raw_name, '[\s,]+(Jr\.?|Sr\.?|II|III|IV)$', '', 'i')
                        FROM '([A-Za-z''\-]+)$')) AS kicker,
        g.game_date,
        coalesce(CASE WHEN raw_num BETWEEN 17 AND 70 THEN raw_num END, kick_distance) AS kick_distance_fixed
    FROM fg
    LEFT JOIN core.games g
        ON g.game_id = fg.game_id
),
kicker_games AS (
    SELECT
        kicker, offense, game_id, game_date
        ,count(*)                                                                       as attempts
        ,SUM(CASE WHEN outcome  = 'fg_made' THEN 1 ELSE 0 END)                          as makes
        ,SUM(CASE WHEN kick_distance_fixed >= 50 THEN 1 ELSE 0 END)                           as long_distance_kick_att
        ,SUM(CASE WHEN kick_distance_fixed >= 50 AND outcome = 'fg_made' THEN 1 ELSE 0 END)   as long_distance_kick_makes
        ,MAX(CASE WHEN outcome = 'fg_made' THEN kick_distance_fixed ELSE 0 END)         as longest_made_kick
    FROM named
    GROUP BY kicker, offense, game_id, game_date
),
kicker_history AS (
    -- each kicker's career totals from EARLIER games only (frame ends one game back).
    -- Keyed on kicker + team, so two kickers with the same key at different schools stay apart.
    SELECT kicker, offense, game_id,
           coalesce(sum(attempts) OVER w, 0)                    AS kicker_att_prior,
           coalesce(sum(makes)    OVER w, 0)                    AS kicker_makes_prior,
           coalesce(sum(long_distance_kick_makes) OVER w, 0)    AS kicker_long_distance_makes,
           coalesce(sum(long_distance_kick_att) OVER w, 0)      AS kicker_long_distance_atts,      
           COALESCE(max(longest_made_kick) OVER w, 0)           AS kicker_longest_kick
    FROM kicker_games
    WINDOW w AS (PARTITION BY kicker, offense ORDER BY game_date, game_id
                 ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING)
)

SELECT
    named.*,
    CASE WHEN raw_num BETWEEN 17 AND 70 THEN raw_num END                          AS yard_num,
    
    h.kicker_att_prior,
    h.kicker_makes_prior,
    COALESCE(h.kicker_makes_prior  / NULLIF(h.kicker_att_prior,0),0)                                     AS kicker_all_time_pct,
    h.kicker_long_distance_atts,
    h.kicker_long_distance_makes,
    COALESCE(h.kicker_long_distance_makes  / NULLIF(h.kicker_long_distance_atts,0),0)            AS kicker_long_distance_pct,
    h.kicker_longest_kick
    
FROM named
LEFT JOIN kicker_history h
    ON h.kicker = named.kicker AND h.offense = named.offense AND h.game_id = named.game_id
-- WHERE h.kicker IS NOT NULL
ORDER BY h.kicker_makes_prior DESC


"""

fg = pd.read_sql(fg_query, engine)
fg

,play_id,game_id,drive_id,season,week,offense,defense,decision,distance,yards_to_goal,...,game_date,kick_distance_fixed,yard_num,kicker_att_prior,kicker_makes_prior,kicker_all_time_pct,kicker_long_distance_atts,kicker_long_distance_makes,kicker_long_distance_pct,kicker_longest_kick
0,252440248031,252440248,25244024805,2005,1,Oregon,Houston,field goal,12,24,...,2005-09-01,39,39.0,NaN,NaN,0.0,NaN,NaN,0.0,NaN
1,2128601530801,212860153,21286015308,2001,8,Virginia,North Carolina,field goal,6,28,...,2001-10-13,45,NaN,NaN,NaN,0.0,NaN,NaN,0.0,NaN
2,-18936,401302348,-2429,2020,3,New Mexico State,Tarleton State,field goal,9,9,...,None,27,27.0,NaN,NaN,0.0,NaN,NaN,0.0,NaN
3,272792440117,272792440,27279244015,2007,6,Nevada,Fresno State,field goal,23,29,...,2007-10-06,46,NaN,NaN,NaN,0.0,NaN,NaN,0.0,NaN
4,401856678258,401856678,4018566789,2026,2,Missouri,Kansas,field goal,10,10,...,2026-09-11,27,NaN,NaN,NaN,0.0,NaN,NaN,0.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
53849,400756907101929001,400756907,4007569074,2015,2,Clemson,App State,field goal,10,21,...,2015-09-12,39,39.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
53850,400756907104989801,400756907,40075690732,2015,2,Clemson,App State,field goal,6,30,...,2015-09-12,47,47.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
53851,400756907104854401,400756907,40075690726,2015,2,Clemson,App State,field goal,4,20,...,2015-09-12,37,37.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
53852,2124402593207,212440259,21244025932,2001,2,Virginia Tech,UConn,field goal,8,30,...,2001-09-01,47,47.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [4]:
fg.info()

<class 'pandas.DataFrame'>
RangeIndex: 53854 entries, 0 to 53853
Data columns (total 43 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   play_id                     53854 non-null  str    
 1   game_id                     53854 non-null  int64  
 2   drive_id                    53854 non-null  str    
 3   season                      53854 non-null  int64  
 4   week                        53854 non-null  int64  
 5   offense                     53854 non-null  str    
 6   defense                     53854 non-null  str    
 7   decision                    53854 non-null  str    
 8   distance                    53854 non-null  int64  
 9   yards_to_goal               53854 non-null  int64  
 10  kick_distance               53854 non-null  int64  
 11  is_goal_to_go               53854 non-null  bool   
 12  period                      53854 non-null  int64  
 13  is_overtime                 53854 non-null

In [12]:
# Create binary target variable
fg['fg_made'] = np.where(fg['outcome'] == 'fg_made', 1, 0)

# Drop identifer columns
df = fg.drop(columns=['play_id','game_id','drive_id','outcome','offense','defense','decision','raw_num','yard_num','raw_name','kicker','game_date'],errors='ignore')

# Seasons for the train / validation / test split
training_season = range(2014,2024)
validate_season = 2024
test_season = 2025

league_rate = df[df['season'].isin(training_season)]['fg_made'].mean()
attempts_so_far = df['fg_att_szn'].fillna(0)
makes_so_far = (df['fg_pct_szn'] * df['fg_att_szn']).fillna(0)
df['fg_shrunk_rate'] = (makes_so_far + 10 * league_rate) / (attempts_so_far + 10)

# Season x distance interaction: lets the improvement over time differ by distance
is_train = df['season'].isin(training_season)
season_c = df['season'] - df.loc[is_train, 'season'].mean()
distance_c = df['kick_distance_fixed'] - df.loc[is_train, 'kick_distance_fixed'].mean()
df['season_x_distance'] = season_c * distance_c

# Group columns by data type for pipelines
numerical_columns = df.select_dtypes(include='number').columns.to_list()
bool_columns = df.select_dtypes(include=['bool','object']).columns.to_list()

not_features = ['fg_made','kick_distance','yards_to_goal','elo_diff','week','kick_distance_fixed','season_x_distance']
numerical_features = [c for c in numerical_columns if c not in not_features]

df[bool_columns] = df[bool_columns].astype('float')

# Split out target and features and create train test splits
X_all = df.drop(columns='fg_made')
y_all = df['fg_made']

train = df[df['season'].isin(training_season)]
valid = df[df['season'] == validate_season]
test = df[df['season'] == test_season]

# Valide the groups
print(train.shape)
print(valid.shape)
print(test.shape)

train.columns

(23927, 34)
(2628, 34)
(2660, 34)


Index(['season', 'week', 'distance', 'yards_to_goal', 'kick_distance',
       'is_goal_to_go', 'period', 'is_overtime', 'sec_left_in_half',
       'sec_left_in_game', 'score_margin', 'is_home', 'is_neutral',
       'off_ppa_szn', 'off_success_szn', 'def_ppa_szn', 'off_elo', 'def_elo',
       'elo_diff', 'offense_spread', 'fg_pct_szn', 'fg_att_szn',
       'off_games_prior', 'kick_distance_fixed', 'kicker_att_prior',
       'kicker_makes_prior', 'kicker_all_time_pct',
       'kicker_long_distance_atts', 'kicker_long_distance_makes',
       'kicker_long_distance_pct', 'kicker_longest_kick', 'fg_made',
       'fg_shrunk_rate', 'season_x_distance'],
      dtype='str')

In [6]:
# Check the shrunk rate: should run roughly 0.5-0.92 with no missing values
print(f"league rate: {league_rate:.3f}")
df['fg_shrunk_rate'].describe()

league rate: 0.745


count    53854.000000
mean         0.748733
std          0.065534
min          0.402369
25%          0.709605
50%          0.744974
75%          0.791161
max          0.937799
Name: fg_shrunk_rate, dtype: float64

In [7]:
# Create Pipelines to prepare data for training
numerical_pipeline = make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler(),
)

categorical_pipeline = make_pipeline(
    SimpleImputer(strategy='constant', fill_value=0)
)

preprocessor = ColumnTransformer([
    ('numerics', numerical_pipeline, numerical_features),
    ('binary',categorical_pipeline, bool_columns),
    ('distance',SplineTransformer(extrapolation='linear'), ['kick_distance_fixed']),
    ('week', SplineTransformer(extrapolation='constant', n_knots=4), ['week'])
])

logistic_regression_pipeline = Pipeline([
    ('preprocess',preprocessor),
    ('model', LogisticRegression(
        max_iter=1000))]
)

# Prepare training data
feature_columns = numerical_features + bool_columns  + ['kick_distance_fixed'] + ['week']
X_train, y_train = train[feature_columns], train['fg_made']
X_valid, y_valid = valid[feature_columns], valid['fg_made']
X_test, y_test = test[feature_columns], test['fg_made']

# Fit and logistic regression model
logistic_regression_pipeline.fit(X_train, y_train)
valid_prob = logistic_regression_pipeline.predict_proba(X_valid)[:,1]
y_pred = logistic_regression_pipeline.predict(X_valid)

In [8]:
print(
    classification_report(
        y_valid,
        y_pred,
        labels=[0, 1],
        target_names=["Field Goal Missed", "Field Goal Made"],
        digits=4
    )
)

accuracy = accuracy_score(y_valid, y_pred)

# Metrics for the positive class: Default = 1
precision = precision_score(y_valid, y_pred, pos_label=1)
sensitivity = recall_score(y_valid, y_pred, pos_label=1)
f1 = f1_score(y_valid, y_pred, pos_label=1)
log_loss_score =  log_loss(y_valid, valid_prob)

auc = roc_auc_score(y_valid, valid_prob)

print(f"Accuracy       : {accuracy:.4f}")
print(f"Precision      : {precision:.4f}")
print(f"Default Recall : {sensitivity:.4f}")
print(f"F1 Score       : {f1:.4f}")
print(f"ROC AUC        : {auc:.4f}")
print(f"Log Loss       : {log_loss_score:4f}")
print("\n\n")

                   precision    recall  f1-score   support

Field Goal Missed     0.4778    0.1361    0.2118       632
  Field Goal Made     0.7770    0.9529    0.8560      1996

         accuracy                         0.7565      2628
        macro avg     0.6274    0.5445    0.5339      2628
     weighted avg     0.7050    0.7565    0.7011      2628

Accuracy       : 0.7565
Precision      : 0.7770
Default Recall : 0.9529
F1 Score       : 0.8560
ROC AUC        : 0.7064
Log Loss       : 0.503560





# Model vs Baseline
*How does the model performance compare to historical data*

In [9]:
# ---------- lookup table: historical make rate by distance band (training seasons only) ----------
band_edges = [0, 22, 27, 32, 37, 42, 47, 52, 57, 100]          # 5-yard bands; 57+ is open-ended (few long kicks)
band_labels = ['<22', '22-26', '27-31', '32-36', '37-41', '42-46', '47-51', '52-56', '57+']

def distance_band(kick_distance):
    return pd.cut(kick_distance, bins=band_edges, labels=band_labels, right=False)

# make rate per band, learned from the training seasons (corrected kick distances)
lookup = (
    train.assign(band=distance_band(train['kick_distance_fixed']))
         .groupby('band', observed=True)['fg_made']
         .agg(make_rate='mean', kicks='size')
)
display(lookup.round(3))

# each validation kick gets its band's training rate (overall training rate if a band is missing)
lookup_prob = (
    distance_band(valid['kick_distance_fixed'])
        .map(lookup['make_rate'])
        .astype(float)
        .fillna(y_train.mean())
        .clip(0.01, 0.99)                                    # a 0% or 100% band would make log loss infinite
        .to_numpy()
)

# ---------- compare on the same validation kicks ----------
def evaluate(name, y_true, prob):
    return {
        'model': name,
        'log_loss': log_loss(y_true, prob),
        'brier': brier_score_loss(y_true, prob),
        'auc': roc_auc_score(y_true, prob),
        'mean_pred': prob.mean(),
        'actual_rate': y_true.mean(),
    }

naive_prob = np.full(len(y_valid), y_train.mean())

comparison = pd.DataFrame([
    evaluate('naive (training rate)', y_valid, naive_prob),
    evaluate('lookup table (5-yd bands)', y_valid, lookup_prob),
    evaluate('logistic + spline', y_valid, valid_prob),
]).set_index('model')
display(comparison.round(4))

# ---------- calibration by distance band (validation season) ----------
# gap = actual - predicted, in percentage points (+ = predicted too low).
# Goal: within 3 points up to 49 yards, 5 points from 50 on (fewer kicks out there).
# std_err = rough noise in the actual rate for that many kicks; a gap well inside it may be luck.
calib_edges = [0, 30, 35, 40, 45, 50, 55, 100]
calib_labels = ['<30', '30-34', '35-39', '40-44', '45-49', '50-54', '55+']

band_check = (
    valid.assign(model=valid_prob, lookup=lookup_prob,
                 band=pd.cut(valid['kick_distance_fixed'], calib_edges, labels=calib_labels, right=False))
         .groupby('band', observed=True)
         .agg(kicks=('fg_made', 'size'), actual=('fg_made', 'mean'),
              model=('model', 'mean'), lookup=('lookup', 'mean'))
)
band_check['model_gap_pts'] = (band_check['actual'] - band_check['model']) * 100
band_check['lookup_gap_pts'] = (band_check['actual'] - band_check['lookup']) * 100
band_check['std_err_pts'] = np.sqrt(band_check['actual'] * (1 - band_check['actual']) / band_check['kicks']) * 100
band_check['tolerance_pts'] = [3, 3, 3, 3, 3, 5, 5]
band_check['model_within'] = band_check['model_gap_pts'].abs() <= band_check['tolerance_pts']

band_check.style.format({
    'actual': '{:.1%}', 'model': '{:.1%}', 'lookup': '{:.1%}',
    'model_gap_pts': '{:+.1f}', 'lookup_gap_pts': '{:+.1f}',
    'std_err_pts': '{:.1f}', 'tolerance_pts': '{:.0f}',
})

,make_rate,kicks
band,,
<22,0.942,1271
22-26,0.915,3779
27-31,0.853,3836
32-36,0.782,3969
37-41,0.711,3973
42-46,0.611,3642
47-51,0.533,2566
52-56,0.431,847
57+,0.318,44


,log_loss,brier,auc,mean_pred,actual_rate
model,,,,,
naive (training rate),0.5522,0.1829,0.5000,0.7450,0.7595
lookup table (5-yd bands),0.5146,0.1705,0.6905,0.7205,0.7595
logistic + spline,0.5036,0.1663,0.7064,0.7513,0.7595


,kicks,actual,model,lookup,model_gap_pts,lookup_gap_pts,std_err_pts,tolerance_pts,model_within
band,,,,,,,,,
<30,722,91.0%,91.8%,89.8%,-0.8,+1.2,1.1,3,True
30-34,380,83.9%,83.5%,80.5%,+0.5,+3.5,1.9,3,True
35-39,403,75.4%,76.4%,73.9%,-0.9,+1.6,2.1,3,True
40-44,440,69.1%,69.6%,64.9%,-0.5,+4.2,2.2,3,True
45-49,385,63.4%,60.7%,56.9%,+2.6,+6.5,2.5,3,True
50-54,237,57.0%,51.0%,47.9%,+6.0,+9.1,3.2,5,False
55+,61,54.1%,43.2%,38.3%,+10.9,+15.8,6.4,5,False


# Recency Weighting

In [10]:
from sklearn.base import clone

weighted_rows = []
for decay in [1.0, 0.9, 0.8, 0.7]:
    weights =  decay ** (train['season'].max() - train['season'])
    weighted_model  = clone(logistic_regression_pipeline)
    weighted_model.fit(X_train, y_train, model__sample_weight=weights)

    prob = weighted_model.predict_proba(X_valid)[:,1]     # score the validation season
    weighted_rows.append(evaluate(f'logistic + spline, decay {decay}', y_valid, prob))

pd.concat([comparison, pd.DataFrame(weighted_rows).set_index('model')]).round(4)

,log_loss,brier,auc,mean_pred,actual_rate
model,,,,,
naive (training rate),0.5522,0.1829,0.5000,0.7450,0.7595
lookup table (5-yd bands),0.5146,0.1705,0.6905,0.7205,0.7595
logistic + spline,0.5036,0.1663,0.7064,0.7513,0.7595
"logistic + spline, decay 1.0",0.5036,0.1663,0.7064,0.7513,0.7595
"logistic + spline, decay 0.9",0.5034,0.1662,0.7063,0.7520,0.7595
"logistic + spline, decay 0.8",0.5036,0.1662,0.7058,0.7523,0.7595
"logistic + spline, decay 0.7",0.5039,0.1663,0.7051,0.7522,0.7595


# Final Model

In [11]:
# Final Model
# Design was chosen on the 2024 validation season; now refit on everything before the
# test season (2014-2024) and score 2025 once. clone() makes a fresh copy, so the
# validated logistic_regression_pipeline above stays as it was.
from sklearn.base import clone

final_train = pd.concat([train, valid])
X_final_train, y_final_train = final_train[feature_columns], final_train['fg_made']

final_model = clone(logistic_regression_pipeline)
final_model.fit(X_final_train, y_final_train)

final_y_pred = final_model.predict(X_test)
final_y_proba = final_model.predict_proba(X_test)[:,1]

print(
    classification_report(
        y_test,
        final_y_pred,
        labels=[0, 1],
        target_names=["Field Goal Missed", "Field Goal Made"],
        digits=4
    )
)

accuracy = accuracy_score(y_test, final_y_pred)

# Metrics for the positive class: Made = 1
precision = precision_score(y_test, final_y_pred, pos_label=1)
sensitivity = recall_score(y_test, final_y_pred, pos_label=1)
f1 = f1_score(y_test, final_y_pred, pos_label=1)
log_loss_score = log_loss(y_test, final_y_proba)

auc = roc_auc_score(y_test, final_y_proba)

print(f"Test season    : {test_season} (trained on {min(training_season)}-{validate_season})")
print(f"Accuracy       : {accuracy:.4f}")
print(f"Precision      : {precision:.4f}")
print(f"Made Recall    : {sensitivity:.4f}")
print(f"F1 Score       : {f1:.4f}")
print(f"ROC AUC        : {auc:.4f}")
print(f"Log Loss       : {log_loss_score:.4f}")
print(f"Mean predicted : {final_y_proba.mean():.3f}  vs actual {y_test.mean():.3f}")

                   precision    recall  f1-score   support

Field Goal Missed     0.5390    0.1277    0.2065       595
  Field Goal Made     0.7940    0.9685    0.8726      2065

         accuracy                         0.7805      2660
        macro avg     0.6665    0.5481    0.5396      2660
     weighted avg     0.7369    0.7805    0.7236      2660

Test season    : 2025 (trained on 2014-2024)
Accuracy       : 0.7805
Precision      : 0.7940
Made Recall    : 0.9685
F1 Score       : 0.8726
ROC AUC        : 0.7221
Log Loss       : 0.4779
Mean predicted : 0.758  vs actual 0.776


# Conclusion

**1. Purpose**

Creation a model that predicts the probability that a field goal is made. Intended to be an input of a 4th-down decision tool (kick vs go vs punt)

**2. Data**

Source: `marts.fourth_down_features` - field goal attempts
Split: Trained on 2014-2023 yearly seasons, validated on 2024 season, tested once on 2025 season.
Notes: roughly 270 kicks had the wrong distance. Original distance was equal to `yards_to_go` + 17 (to account for length of endzone). as a result, kick distance was stripped from the play text.

    ex.(Daniel Carlson 48 yard kick)


**3. Results  (2025 Test Season)**

Final model refit on 2014-2024, scored once on the 2025 season (2,660 kicks).

| Metric | Value |
|---|---|
| Log loss | 0.478 |
| ROC AUC | 0.721 |
| Mean predicted make rate | 76.2% |
| Actual make rate | 77.6% |

**Calibration by distance** (gap = actual − predicted, in percentage points; noise = rough ± for that many kicks):

| Distance | Kicks | Actual | Predicted | Gap | Noise (±) |
|---|---|---|---|---|---|
| < 30 | 734 | 92.0% | 91.8% | +0.1 | 1.0 |
| 30–34 | 370 | 87.3% | 84.2% | +3.1 | 1.7 |
| 35–39 | 430 | 80.9% | 77.4% | +3.5 | 1.9 |
| 40–44 | 421 | 73.2% | 70.9% | +2.3 | 2.2 |
| 45–49 | 421 | 60.6% | 62.3% | −1.7 | 2.4 |
| 50–54 | 221 | 57.5% | 53.8% | +3.7 | 3.3 |
| 55+ | 63 | 46.0% | 47.8% | −1.7 | 6.3 |

- The overall make rate is within 1.5 points (76.2% predicted vs 77.6% actual).
- Long kicks are within the ±5-point goal (50–54: +3.7, 55+: −1.7). On the 2024 validation season, before 2024 was in training, they were off by 6–12 points; refitting on 2014-2024 closed most of that gap.
- 30–39 yards runs about 3 points low, slightly outside the ±3-point goal. 2025 was the best kicking season in the data, so the model trails the trend a little on a season it hasn't seen.
- On the 2024 validation season the model beat the distance lookup table (log loss 0.504 vs 0.515) and was closer to the actual make rate in every distance band.


**4. What Worked vs What Didn't**

| Change | Effect | Implemented |
|---|---|---|
|Spline on distance| followed curve| Yes|
|season as feature | fixed underprediction| Yes|
|Distance Fixed | small log loss change, removed impossible kicks (109 kicks)| Yes|
|Shrunk Kicking % | no measurable gain | No |
|Recency Weighting | no gain once season was added | No|
|Season x Distance Feature | No Gain | No |


**5. Limitations**
- No Kicker Identity (built off team percentages)
- no weather or wind features
- very few long kicks

**6. Possible Improvements**
- Goal: make above expected
- Add in kicker identities.

**7. Final Notes**
First model created with new database. The database it self suffers from complexity which can make it hard to engineer features (namely kicker identity). Model could be improved but will move on for now, as it is performing at goal (within 3 points of expected)
